# Rough likelihoods break gradient samplers — $d=100$

The true target is $N(0, I_{100})$. The sampler sees an **isotropically** perturbed log-density

$$\log\pi(\theta) = -\tfrac12\|\theta\|^2 + A\sum_{i=1}^{100}\sin(\omega\theta_i),$$

with the exact path-derivative gradient. We assume isotropy, so $A$ and $\omega$ are scalars and the grid search is again 2-D over $(A,\omega)$. WALNUTS is the kernel from the `DanWaxman/blackjax` fork. Divergence-to-truth uses the **diagonal** moment-matched Gaussian KL, $\tfrac12\sum_i(\hat\sigma_i^2 + \hat\mu_i^2 - 1 - \log\hat\sigma_i^2)$ — appropriate here because the per-coordinate sinusoid induces no cross-correlations, and far better conditioned than a full $100\times100$ covariance fit.

In [ ]:
import time
import warnings

warnings.filterwarnings("ignore")

import arviz as az
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from tqdm import tqdm

import manual_gaussian_inference as mgi

%matplotlib inline

DIM = 100
NUM_WARMUP = 2000
NUM_SAMPLES = 5000
N_REPS = 3
PROFILE_A = 0.1
PROFILE_OMEGA = 10.0

In [ ]:
# "Data" / ground truth: iid draws from the true N(0, I_100).
system_truth = mgi.make_perturbed_gaussian_system(DIM)
truth = mgi.simulate_true_samples(jr.PRNGKey(0), system_truth, 2000)

## Profile likelihoods

1-D profile (coordinate 0, others fixed at 0) under a few $(A,\omega)$ — the perturbed target factorizes across coordinates, so this is the $d=1$ likelihood shape — and a 2-D profile (coordinates 0–1) at the representative $(A,\omega)=(0.1, 10)$.

In [ ]:
# 1-D profile along coordinate 0 (others at 0) under a few (A, omega).
grid = jnp.linspace(-2.0, 2.0, 500)
settings = [(0.1, 10.0), (0.1, 100.0)]


def _coord0_logdensity(system, x):
    return mgi.perturbed_log_density(system, jnp.zeros(DIM).at[0].set(x))


true_curve = jax.vmap(
    lambda x: mgi.true_log_density(system_truth, jnp.zeros(DIM).at[0].set(x))
)(grid)
curves, labels = [], []
for a_val, w_val in settings:
    system = mgi.make_perturbed_gaussian_system(DIM, a_val, w_val)
    curves.append(jax.vmap(lambda x: _coord0_logdensity(system, x))(grid))
    labels.append(f"A={a_val}, ω={w_val}")
fig, ax = mgi.plot_likelihood_curves(grid, true_curve, curves, labels=labels)
plt.show()

In [ ]:
# 2-D profile: vary coords 0 and 1, all others fixed at 0.
profile_system = mgi.make_perturbed_gaussian_system(DIM, PROFILE_A, PROFILE_OMEGA)
gx = jnp.linspace(-2.0, 2.0, 120)
gy = jnp.linspace(-2.0, 2.0, 120)


def _profile_2d(x, y):
    theta = jnp.zeros(DIM).at[0].set(x).at[1].set(y)
    return mgi.perturbed_log_density(profile_system, theta)


profile_grid = jax.vmap(lambda x: jax.vmap(lambda y: _profile_2d(x, y))(gy))(gx)
fig, _ = mgi.plot_profile_2d(gx, gy, profile_grid, coord_indices=(0, 1))
plt.show()

## Grid search over $(A, \omega)$

Each cell averages diagonal-KL-to-truth and ESS/s over `N_REPS` independent runs, for NUTS and WALNUTS.

In [ ]:
a_grid = np.logspace(-5.0, -1.0, 5, base=10)
omega_grid = np.linspace(5.0, 500.0, 10)

In [ ]:
def ess_min(samples):
    """Minimum effective sample size across coordinates (worst direction)."""
    arr = np.asarray(samples)
    return min(float(np.asarray(az.ess(arr[None, :, d]))) for d in range(arr.shape[1]))


def run_repeated(sampler, logdensity_fn, divergence_fn):
    """Run a sampler over N_REPS seeds; return (mean divergence, mean ESS/s, pooled
    samples). Wall-clock per run is end-to-end (JAX compile + warmup + sampling)."""
    divs, ess_rates, pooled = [], [], []
    for rep in range(N_REPS):
        t0 = time.perf_counter()
        result = sampler(
            logdensity_fn, rep, dim=DIM,
            num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
        )
        samples = np.asarray(jax.block_until_ready(result["samples"]))
        dt = time.perf_counter() - t0
        divs.append(divergence_fn(samples))
        ess_rates.append(ess_min(samples) / dt)
        pooled.append(samples)
    return float(np.mean(divs)), float(np.mean(ess_rates)), np.concatenate(pooled, 0)

In [ ]:
# Coordinate-0 marginal of the pooled samples vs. true N(0, 1) and the perturbed
# marginal ("ML"). The d=100 target factorizes across coordinates, so the marginal of
# any single coordinate is the 1-D perturbed density (== true N(0,1) when A=0).
cases = [(0.0, 0.0), (1e-2, 50.0), (1e-1, 100.0)]
hist_grid = jnp.linspace(-4.0, 4.0, 600)
hist_grid_np = np.asarray(hist_grid)
true_pdf = norm.pdf(hist_grid_np)
bins = np.linspace(-4.0, 4.0, 61)

fig, axes = plt.subplots(
    len(cases),
    2,
    figsize=(11, 2.8 * len(cases)),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for row, (A, omega) in enumerate(cases):
    system = mgi.make_perturbed_gaussian_system(DIM, A, omega)
    logdensity_fn = mgi.make_blackjax_logdensity(system, mode="perturbed")

    marginal = mgi.make_perturbed_gaussian_system(1, A, omega)
    log_pert = np.asarray(
        jax.vmap(lambda t: mgi.perturbed_log_density(marginal, t[None]))(hist_grid)
    )
    dens = np.exp(log_pert - log_pert.max())
    pert_pdf = dens / np.trapezoid(dens, hist_grid_np)

    _, _, nuts_final = run_repeated(
        mgi.run_blackjax_nuts, logdensity_fn, lambda s: 0.0
    )
    _, _, walnuts_final = run_repeated(mgi.run_walnuts, logdensity_fn, lambda s: 0.0)

    for ax, samples, name in (
        (axes[row, 0], nuts_final, "NUTS"),
        (axes[row, 1], walnuts_final, "WALNUTS"),
    ):
        ax.hist(
            np.asarray(samples)[:, 0], bins=bins, density=True, color="C0", alpha=0.4
        )
        ax.plot(hist_grid_np, true_pdf, color="0.2", linewidth=2.5, label="true $N(0,1)$")
        ax.plot(
            hist_grid_np,
            pert_pdf,
            color="C3",
            linewidth=2.5,
            linestyle="--",
            label="perturbed marginal (ML)",
        )
        ax.set_title(f"{name}  (A={A}, ω={omega})  coord 0")
        ax.set_yticks([])
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)

axes[0, 0].set_xlim(-4.0, 4.0)
axes[0, 0].legend(loc="upper right", frameon=False, fontsize=8)
plt.show()

In [ ]:
n_a, n_w = len(a_grid), len(omega_grid)
div_nuts = np.full((n_a, n_w), np.nan)
div_walnuts = np.full((n_a, n_w), np.nan)
ess_nuts = np.full((n_a, n_w), np.nan)
ess_walnuts = np.full((n_a, n_w), np.nan)
divergence_fn = lambda s: float(mgi.gaussian_kl_to_standard_normal(s))

for i_a in tqdm(range(n_a)):
    for j_w in tqdm(range(n_w), leave=False):
        system = mgi.make_perturbed_gaussian_system(
            DIM, float(a_grid[i_a]), float(omega_grid[j_w])
        )
        logdensity_fn = mgi.make_blackjax_logdensity(system, mode="perturbed")
        div_nuts[i_a, j_w], ess_nuts[i_a, j_w], _ = run_repeated(
            mgi.run_blackjax_nuts, logdensity_fn, divergence_fn
        )
        div_walnuts[i_a, j_w], ess_walnuts[i_a, j_w], _ = run_repeated(
            mgi.run_walnuts, logdensity_fn, divergence_fn
        )

In [ ]:
fig, _ = mgi.plot_nuts_walnuts_heatmaps(
    a_grid, omega_grid, div_nuts, div_walnuts,
    quantity_label=f"diagonal KL to N(0, I) ({N_REPS}-run mean)", log_color=True,
)
plt.show()

In [ ]:
fig, _ = mgi.plot_nuts_walnuts_heatmaps(
    a_grid, omega_grid, ess_nuts, ess_walnuts,
    quantity_label=f"ESS / s ({N_REPS}-run mean)", log_color=False,
)
plt.show()

## Pooled NUTS samples vs. truth at $(A,\omega)=(0.1, 10.0)$, first four coordinates

In [ ]:
final_logdensity = mgi.make_blackjax_logdensity(profile_system, mode="perturbed")
_, _, nuts_final = run_repeated(mgi.run_blackjax_nuts, final_logdensity, lambda s: 0.0)

In [ ]:
pair = mgi.plot_pairplot_vs_truth(nuts_final, np.asarray(truth), coords=(0, 1, 2, 3))
plt.show()